In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [6]:
# """
# =============================================================================
# Federated Learning + Differential Privacy vs Gradient Inversion
# =============================================================================

# Design project implementation.

# Arms:
#   A. FedAvg              - no DP (control)
#   B. LDP-Fed style       - fixed global clipping + plain Gaussian noise
#   C. ALDP-FL style       - adaptive per-layer clipping (sliding window k=3)
#                            + bounded (truncated) Gaussian noise

# Experiments:
#   1. Attack at round 10 under A, B(eps 10/5/1), C(eps 10/5/1)   -> 7 runs
#   2. Attack under A at round 1 vs round 10                      -> training-stage effect
#   3. Attack the accumulated 2-epoch update instead of a gradient-> expected failure
#   4. Test accuracy per round for A, B(eps=5), C(eps=5)
#   5. Wall-clock overhead of the DP step, fixed vs adaptive

# Outputs:
#   fig_reconstructions.png
#   fig_accuracy.png
#   results_table.txt

# Run on Kaggle with GPU P100 + Internet ON.
# =============================================================================
# """

# import time
# import copy
# import numpy as np
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt
# from torchvision import datasets, transforms
# from torch.utils.data import DataLoader, Subset
# from skimage.metrics import structural_similarity as ssim_fn

# # =============================================================================
# # CONFIG  -- edit these
# # =============================================================================
# DATASET       = "CIFAR10"     # "CIFAR10" or "MNIST"
# N_CLIENTS     = 3
# ROUNDS        = 10
# LOCAL_EPOCHS  = 2
# BATCH_SIZE    = 32
# LR            = 0.05          # SGD, not Adam (Adam breaks invertibility)
# SAMPLES_PER_CLIENT = 2000     # subsample for speed; raise if you have time
# FIXED_CLIP_C  = 4.0           # threshold for Arm B -- TUNE THIS
# ADAPTIVE_ETA  = 0.5           # bounded-perturbation hyperparameter
# DELTA         = 1e-5
# EPS_LIST      = [10, 5, 1]
# ATTACK_ITERS  = 200           # LBFGS outer steps
# VICTIM_IDX    = 25            # which image to attack
# SEED          = 0

# torch.manual_seed(SEED)
# np.random.seed(SEED)
# DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"device: {DEV}")

# IN_CH    = 3 if DATASET == "CIFAR10" else 1
# IMG_SIZE = 32 if DATASET == "CIFAR10" else 28
# N_CLASS  = 10


# # =============================================================================
# # MODEL
# # LeNet with SIGMOID activations. Required: DLG differentiates through the
# # gradient, so the activation must be twice differentiable. ReLU breaks this.
# # =============================================================================
# class LeNet(nn.Module):
#     def __init__(self):
#         super().__init__()
#         act = nn.Sigmoid
#         self.body = nn.Sequential(
#             nn.Conv2d(IN_CH, 12, 5, padding=2, stride=2), act(),
#             nn.Conv2d(12, 12, 5, padding=2, stride=2), act(),
#             nn.Conv2d(12, 12, 5, padding=2, stride=1), act(),
#         )
#         feat = 12 * (IMG_SIZE // 4) * (IMG_SIZE // 4)
#         self.fc = nn.Linear(feat, N_CLASS)

#     def forward(self, x):
#         out = self.body(x)
#         return self.fc(out.view(out.size(0), -1))


# def weights_init(m):
#     if hasattr(m, "weight") and m.weight is not None:
#         m.weight.data.uniform_(-0.5, 0.5)
#     if hasattr(m, "bias") and m.bias is not None:
#         m.bias.data.uniform_(-0.5, 0.5)


# # =============================================================================
# # DATA
# # =============================================================================
# def load_data():
#     tf = transforms.ToTensor()
#     if DATASET == "CIFAR10":
#         train = datasets.CIFAR10("./data", train=True,  download=True, transform=tf)
#         test  = datasets.CIFAR10("./data", train=False, download=True, transform=tf)
#     else:
#         train = datasets.MNIST("./data", train=True,  download=True, transform=tf)
#         test  = datasets.MNIST("./data", train=False, download=True, transform=tf)

#     idx = np.random.permutation(len(train))
#     client_sets = []
#     for c in range(N_CLIENTS):
#         s = idx[c * SAMPLES_PER_CLIENT:(c + 1) * SAMPLES_PER_CLIENT]
#         client_sets.append(Subset(train, s.tolist()))

#     test_loader = DataLoader(Subset(test, list(range(2000))), batch_size=256)
#     return client_sets, test_loader, train


# # =============================================================================
# # DP MECHANISMS
# # =============================================================================
# def gaussian_sigma(sensitivity, eps, delta=DELTA):
#     """Standard Gaussian mechanism:  sigma = Df * sqrt(2 ln(1.25/delta)) / eps"""
#     return sensitivity * np.sqrt(2.0 * np.log(1.25 / delta)) / eps


# def dp_fixed(tensors, eps, C=FIXED_CLIP_C):
#     """
#     Arm B -- LDP-Fed style.
#     ONE global clipping threshold for the whole update, plain Gaussian noise.
#     """
#     total = torch.sqrt(sum((t ** 2).sum() for t in tensors))
#     scale = min(1.0, C / (total.item() + 1e-12))
#     sigma = gaussian_sigma(C, eps)
#     return [t * scale + torch.randn_like(t) * sigma for t in tensors]


# def dp_adaptive(tensors, history, eps, eta=ADAPTIVE_ETA):
#     """
#     Arm C -- ALDP-FL style.
#       - per-layer threshold = mean of that layer's last 3 recorded L2 norms
#       - Gaussian noise truncated to [-b, b], b = (e^eps - eta)/(e^eps + eta)
#     history: list-of-lists, one per layer, mutated in place.
#     """
#     b = (np.exp(eps) - eta) / (np.exp(eps) + eta)
#     out = []
#     for i, t in enumerate(tensors):
#         n = t.norm().item()
#         history[i].append(n)
#         history[i][:] = history[i][-3:]              # sliding window k = 3
#         C_layer = float(np.mean(history[i])) + 1e-12
#         scale = min(1.0, C_layer / (n + 1e-12))
#         sigma = gaussian_sigma(C_layer, eps)
#         noise = torch.clamp(torch.randn_like(t) * sigma, -b, b)
#         out.append(t * scale + noise)
#     return out


# # =============================================================================
# # LOCAL TRAINING
# # =============================================================================
# def local_train(global_model, dataset, dp_mode, eps, history, timer):
#     """
#     One client's local training. Returns the update (delta) it would send.
#     DP is applied to the accumulated delta before it leaves the client.
#     """
#     model = copy.deepcopy(global_model).to(DEV)
#     opt = torch.optim.SGD(model.parameters(), lr=LR)
#     crit = nn.CrossEntropyLoss()
#     loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

#     start = [p.detach().clone() for p in global_model.parameters()]

#     model.train()
#     for _ in range(LOCAL_EPOCHS):
#         for xb, yb in loader:
#             xb, yb = xb.to(DEV), yb.to(DEV)
#             opt.zero_grad()
#             crit(model(xb), yb).backward()
#             opt.step()

#     delta = [p.detach() - s for p, s in zip(model.parameters(), start)]

#     if dp_mode == "none":
#         return delta
#     t0 = time.perf_counter()
#     if dp_mode == "fixed":
#         delta = dp_fixed(delta, eps)
#     else:
#         delta = dp_adaptive(delta, history, eps)
#     timer.append(time.perf_counter() - t0)
#     return delta


# def evaluate(model, loader):
#     model.eval()
#     correct = total = 0
#     with torch.no_grad():
#         for xb, yb in loader:
#             xb, yb = xb.to(DEV), yb.to(DEV)
#             correct += (model(xb).argmax(1) == yb).sum().item()
#             total += yb.size(0)
#     return correct / total


# def run_fl(client_sets, test_loader, dp_mode, eps, snapshot_rounds=()):
#     """Full FedAvg loop. Returns accuracy curve, snapshots, DP timing."""
#     model = LeNet().to(DEV)
#     model.apply(weights_init)
#     history = [[] for _ in model.parameters()]
#     timer, acc_curve, snaps = [], [], {}

#     for r in range(1, ROUNDS + 1):
#         deltas = [local_train(model, cs, dp_mode, eps, history, timer)
#                   for cs in client_sets]
#         with torch.no_grad():
#             for i, p in enumerate(model.parameters()):
#                 p += torch.stack([d[i] for d in deltas]).mean(0)

#         acc = evaluate(model, test_loader)
#         acc_curve.append(acc)
#         if r in snapshot_rounds:
#             snaps[r] = copy.deepcopy(model)
#         print(f"  [{dp_mode} eps={eps}] round {r:2d}  acc {acc:.4f}", flush=True)

#     return model, acc_curve, snaps, timer


# # =============================================================================
# # THE ATTACK -- Deep Leakage from Gradients
# # =============================================================================
# def dlg_attack(model, target, shape, iters=ATTACK_ITERS, label=""):
#     """Optimise a dummy image so its gradient matches `target`."""
#     torch.manual_seed(SEED)
#     dummy_x = torch.randn(shape, device=DEV).requires_grad_(True)
#     dummy_y = torch.randn((1, N_CLASS), device=DEV).requires_grad_(True)
#     opt = torch.optim.LBFGS([dummy_x, dummy_y], lr=1.0)
#     target = [t.detach() for t in target]

#     for it in range(iters):
#         def closure():
#             opt.zero_grad()
#             pred = model(dummy_x)
#             loss = torch.mean(torch.sum(
#                 -torch.softmax(dummy_y, -1) * torch.log_softmax(pred, -1), 1))
#             dg = torch.autograd.grad(loss, model.parameters(), create_graph=True)
#             diff = sum(((a - b) ** 2).sum() for a, b in zip(dg, target))
#             diff.backward()
#             return diff
#         try:
#             opt.step(closure)
#         except Exception as e:                  # LBFGS occasionally diverges
#             print(f"    [{label}] LBFGS stopped at iter {it}: {e}")
#             break
#         if it % 50 == 0:
#             print(f"    [{label}] iter {it}", flush=True)
#     return dummy_x.detach()


# def victim_gradient(model, x, y):
#     """The gradient a client would compute on the victim image."""
#     crit = nn.CrossEntropyLoss()
#     loss = crit(model(x), y)
#     return [g.detach().clone()
#             for g in torch.autograd.grad(loss, model.parameters())]


# # =============================================================================
# # METRICS
# # =============================================================================
# def to_img(t):
#     a = t[0].detach().cpu().numpy().transpose(1, 2, 0)
#     if a.shape[2] == 1:
#         a = a[:, :, 0]
#     return np.clip(a, 0, 1)


# def score(orig, recon):
#     o, r = to_img(orig), to_img(recon)
#     mse = float(np.mean((o - r) ** 2))
#     ch = -1 if o.ndim == 3 else None
#     s = float(ssim_fn(o, r, channel_axis=ch, data_range=1.0))
#     return mse, s


# # =============================================================================
# # MAIN
# # =============================================================================
# def main():
#     client_sets, test_loader, train_full = load_data()

#     victim_x = train_full[VICTIM_IDX][0].unsqueeze(0).to(DEV)
#     victim_y = torch.tensor([train_full[VICTIM_IDX][1]], device=DEV)

#     # ---------------------------------------------------------------- Arm A
#     print("\n=== Arm A: FedAvg (no DP) ===")
#     modelA, accA, snapsA, _ = run_fl(client_sets, test_loader,
#                                      "none", None, snapshot_rounds=(1, ROUNDS))

#     # ------------------------------------------------- Arms B and C (utility)
#     print("\n=== Arm B: fixed clipping, eps=5 ===")
#     _, accB, _, timeB = run_fl(client_sets, test_loader, "fixed", 5)

#     print("\n=== Arm C: adaptive clipping, eps=5 ===")
#     _, accC, _, timeC = run_fl(client_sets, test_loader, "adaptive", 5)

#     # =====================================================================
#     # EXPERIMENT 1 -- main result, attack at round 10
#     # The SAME model snapshot is used for every arm, so the only variable
#     # is the DP applied to the gradient.
#     # =====================================================================
#     print("\n=== Experiment 1: attack at round 10 ===")
#     m10 = snapsA[ROUNDS]
#     g_true = victim_gradient(m10, victim_x, victim_y)

#     # populate the sliding window with a few real norms so the adaptive
#     # threshold is based on history, as it would be during training
#     hist = [[] for _ in g_true]
#     for _ in range(3):
#         for i, g in enumerate(g_true):
#             hist[i].append(g.norm().item())

#     runs = [("FedAvg (no DP)", g_true)]
#     for e in EPS_LIST:
#         runs.append((f"Fixed eps={e}", dp_fixed(g_true, e)))
#     for e in EPS_LIST:
#         runs.append((f"Adaptive eps={e}",
#                      dp_adaptive(g_true, [h[:] for h in hist], e)))

#     results, images = [], []
#     for name, g in runs:
#         print(f"  -> {name}")
#         rec = dlg_attack(m10, g, victim_x.shape, label=name)
#         mse, s = score(victim_x, rec)
#         results.append((name, mse, s))
#         images.append((name, rec, s))
#         print(f"     MSE {mse:.4f}  SSIM {s:.4f}")

#     # =====================================================================
#     # EXPERIMENT 2 -- does leakage grow with training?
#     # =====================================================================
#     print("\n=== Experiment 2: round 1 vs round 10 (no DP) ===")
#     m1 = snapsA[1]
#     g1 = victim_gradient(m1, victim_x, victim_y)
#     rec1 = dlg_attack(m1, g1, victim_x.shape, label="round1")
#     mse1, s1 = score(victim_x, rec1)
#     exp2 = [("Round 1, no DP", mse1, s1), ("Round 10, no DP",
#                                            results[0][1], results[0][2])]
#     print(f"  round 1 : MSE {mse1:.4f}  SSIM {s1:.4f}")
#     print(f"  round 10: MSE {results[0][1]:.4f}  SSIM {results[0][2]:.4f}")

#     # =====================================================================
#     # EXPERIMENT 3 -- attack the ACCUMULATED update. Expected to fail.
#     # =====================================================================
#     print("\n=== Experiment 3: accumulated multi-epoch update (no DP) ===")
#     tmp = copy.deepcopy(m10).to(DEV)
#     opt = torch.optim.SGD(tmp.parameters(), lr=LR)
#     crit = nn.CrossEntropyLoss()
#     loader = DataLoader(client_sets[0], batch_size=BATCH_SIZE, shuffle=True)
#     start = [p.detach().clone() for p in m10.parameters()]

#     steps = 0
#     for _ in range(LOCAL_EPOCHS):
#         for xb, yb in loader:
#             xb, yb = xb.to(DEV), yb.to(DEV)
#             opt.zero_grad(); crit(tmp(xb), yb).backward(); opt.step()
#             steps += 1

#     # attacker's best guess: invert one SGD step,  g ~ -(delta)/(lr * steps)
#     delta = [p.detach() - s for p, s in zip(tmp.parameters(), start)]
#     g_acc = [-d / (LR * steps) for d in delta]
#     rec_acc = dlg_attack(m10, g_acc, victim_x.shape, label="accumulated")
#     mse_a, s_a = score(victim_x, rec_acc)
#     print(f"  accumulated: MSE {mse_a:.4f}  SSIM {s_a:.4f}  ({steps} SGD steps)")

#     # =====================================================================
#     # OUTPUT
#     # =====================================================================
#     lines = []
#     lines.append("=" * 60)
#     lines.append(f"DATASET: {DATASET}   clients={N_CLIENTS} rounds={ROUNDS} "
#                  f"local_epochs={LOCAL_EPOCHS} batch={BATCH_SIZE}")
#     lines.append("=" * 60)
#     lines.append("\nEXP 1 -- reconstruction quality (round 10)")
#     lines.append(f"{'Setting':<24}{'MSE':>12}{'SSIM':>12}")
#     lines.append("-" * 48)
#     for n, m, s in results:
#         lines.append(f"{n:<24}{m:>12.4f}{s:>12.4f}")

#     lines.append("\nEXP 2 -- training stage effect")
#     for n, m, s in exp2:
#         lines.append(f"{n:<24}{m:>12.4f}{s:>12.4f}")

#     lines.append("\nEXP 3 -- accumulated update (negative result)")
#     lines.append(f"{'Accumulated delta':<24}{mse_a:>12.4f}{s_a:>12.4f}")

#     lines.append("\nEXP 4 -- final test accuracy")
#     lines.append(f"{'FedAvg (no DP)':<24}{accA[-1]:>12.4f}")
#     lines.append(f"{'Fixed eps=5':<24}{accB[-1]:>12.4f}")
#     lines.append(f"{'Adaptive eps=5':<24}{accC[-1]:>12.4f}")

#     lines.append("\nEXP 5 -- DP overhead per call")
#     lines.append(f"{'Fixed':<24}{np.mean(timeB)*1e6:>10.1f} us")
#     lines.append(f"{'Adaptive':<24}{np.mean(timeC)*1e6:>10.1f} us")
#     lines.append("=" * 60)

#     out = "\n".join(lines)
#     print("\n" + out)
#     with open("results_table.txt", "w") as f:
#         f.write(out)

#     # ---- figure 1: reconstructions ----
#     n = len(images) + 1
#     fig, ax = plt.subplots(1, n, figsize=(2.3 * n, 3.0))
#     ax[0].imshow(to_img(victim_x), cmap=None if IN_CH == 3 else "gray")
#     ax[0].set_title("Original", fontsize=10); ax[0].axis("off")
#     for i, (nm, r, s) in enumerate(images):
#         ax[i + 1].imshow(to_img(r), cmap=None if IN_CH == 3 else "gray")
#         ax[i + 1].set_title(f"{nm}\nSSIM {s:.3f}", fontsize=8)
#         ax[i + 1].axis("off")
#     plt.tight_layout()
#     plt.savefig("fig_reconstructions.png", dpi=150, bbox_inches="tight")

#     # ---- figure 2: accuracy ----
#     plt.figure(figsize=(6, 4))
#     rr = range(1, ROUNDS + 1)
#     plt.plot(rr, accA, "o-", label="FedAvg (no DP)")
#     plt.plot(rr, accB, "s-", label="Fixed clipping, eps=5")
#     plt.plot(rr, accC, "^-", label="Adaptive clipping, eps=5")
#     plt.xlabel("Communication round"); plt.ylabel("Test accuracy")
#     plt.title(f"Utility cost of DP ({DATASET})")
#     plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
#     plt.savefig("fig_accuracy.png", dpi=150)

#     print("\nSaved: fig_reconstructions.png, fig_accuracy.png, results_table.txt")


# if __name__ == "__main__":
#     main()

In [7]:
# """
# =============================================================================
# FL + DP vs Gradient Inversion  --  CORRECTED VERSION
# =============================================================================

# Changes from v1:
#   1. DP is parameterised by NOISE MULTIPLIER z, not by a naive epsilon.
#      The textbook bound sigma = C*sqrt(2 ln(1.25/delta))/eps is far too
#      loose in high dimensions: per-parameter signal after clipping is
#      ~C/sqrt(d) while noise is ~sigma, so the ratio blows up with model
#      size. Standard practice (Abadi et al. 2016) fixes z = sigma/C and
#      accounts epsilon separately via RDP. We report z and do NOT claim
#      epsilon values.
#   2. torch.cuda.synchronize() around timing (GPU calls are async; the old
#      numbers were measuring kernel launches, not execution).
#   3. More clients -- averaging K client updates divides the noise by K.
#   4. Prints a noise/signal diagnostic before running, so you can see
#      whether your z values will actually produce a degradation gradient.
# =============================================================================
# """

# import time, copy
# import numpy as np
# import torch
# import torch.nn as nn
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt
# from torchvision import datasets, transforms
# from torch.utils.data import DataLoader, Subset
# from skimage.metrics import structural_similarity as ssim_fn

# # =============================================================================
# # CONFIG
# # =============================================================================
# DATASET       = "CIFAR10"
# N_CLIENTS     = 5             # more clients -> noise averages down by K
# ROUNDS        = 10
# LOCAL_EPOCHS  = 2
# BATCH_SIZE    = 32
# LR            = 0.05
# SAMPLES_PER_CLIENT = 2000
# FIXED_CLIP_C  = 1.0
# NOISE_BOUND   = 1.0           # truncation bound for adaptive arm
# Z_LIST        = [0.001, 0.005, 0.02]   # noise multipliers (NOT epsilon)
# Z_UTILITY     = 0.005                  # z used for the accuracy curves
# ATTACK_ITERS  = 200
# VICTIM_IDX    = 25
# SEED          = 0

# torch.manual_seed(SEED); np.random.seed(SEED)
# DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"device: {DEV}")

# IN_CH    = 3 if DATASET == "CIFAR10" else 1
# IMG_SIZE = 32 if DATASET == "CIFAR10" else 28
# N_CLASS  = 10


# # =============================================================================
# # MODEL -- sigmoid activations are required (DLG needs 2nd derivatives)
# # =============================================================================
# class LeNet(nn.Module):
#     def __init__(self):
#         super().__init__()
#         act = nn.Sigmoid
#         self.body = nn.Sequential(
#             nn.Conv2d(IN_CH, 12, 5, padding=2, stride=2), act(),
#             nn.Conv2d(12, 12, 5, padding=2, stride=2), act(),
#             nn.Conv2d(12, 12, 5, padding=2, stride=1), act(),
#         )
#         feat = 12 * (IMG_SIZE // 4) * (IMG_SIZE // 4)
#         self.fc = nn.Linear(feat, N_CLASS)

#     def forward(self, x):
#         out = self.body(x)
#         return self.fc(out.view(out.size(0), -1))


# def weights_init(m):
#     if hasattr(m, "weight") and m.weight is not None:
#         m.weight.data.uniform_(-0.5, 0.5)
#     if hasattr(m, "bias") and m.bias is not None:
#         m.bias.data.uniform_(-0.5, 0.5)


# # =============================================================================
# # DATA
# # =============================================================================
# def load_data():
#     tf = transforms.ToTensor()
#     if DATASET == "CIFAR10":
#         train = datasets.CIFAR10("./data", train=True,  download=True, transform=tf)
#         test  = datasets.CIFAR10("./data", train=False, download=True, transform=tf)
#     else:
#         train = datasets.MNIST("./data", train=True,  download=True, transform=tf)
#         test  = datasets.MNIST("./data", train=False, download=True, transform=tf)

#     idx = np.random.permutation(len(train))
#     clients = [Subset(train, idx[c*SAMPLES_PER_CLIENT:(c+1)*SAMPLES_PER_CLIENT].tolist())
#                for c in range(N_CLIENTS)]
#     test_loader = DataLoader(Subset(test, list(range(2000))), batch_size=256)
#     return clients, test_loader, train


# # =============================================================================
# # DP MECHANISMS -- parameterised by noise multiplier z
# # =============================================================================
# def dp_fixed(tensors, z, C=FIXED_CLIP_C):
#     """Arm B -- LDP-Fed style: one global clipping threshold, plain Gaussian."""
#     total = torch.sqrt(sum((t ** 2).sum() for t in tensors))
#     scale = min(1.0, C / (total.item() + 1e-12))
#     sigma = z * C
#     return [t * scale + torch.randn_like(t) * sigma for t in tensors]


# def dp_adaptive(tensors, history, z, b=NOISE_BOUND):
#     """
#     Arm C -- ALDP-FL style:
#       per-layer threshold = mean of that layer's last 3 L2 norms,
#       Gaussian noise truncated to [-b, b].
#     """
#     out = []
#     for i, t in enumerate(tensors):
#         n = t.norm().item()
#         history[i].append(n)
#         history[i][:] = history[i][-3:]            # sliding window k = 3
#         C_layer = float(np.mean(history[i])) + 1e-12
#         scale = min(1.0, C_layer / (n + 1e-12))
#         noise = torch.clamp(torch.randn_like(t) * (z * C_layer), -b, b)
#         out.append(t * scale + noise)
#     return out


# # =============================================================================
# # TRAINING
# # =============================================================================
# def local_train(global_model, dataset, dp_mode, z, history, timer):
#     model = copy.deepcopy(global_model).to(DEV)
#     opt = torch.optim.SGD(model.parameters(), lr=LR)
#     crit = nn.CrossEntropyLoss()
#     loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
#     start = [p.detach().clone() for p in global_model.parameters()]

#     model.train()
#     for _ in range(LOCAL_EPOCHS):
#         for xb, yb in loader:
#             xb, yb = xb.to(DEV), yb.to(DEV)
#             opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()

#     delta = [p.detach() - s for p, s in zip(model.parameters(), start)]
#     if dp_mode == "none":
#         return delta

#     if DEV.type == "cuda":
#         torch.cuda.synchronize()
#     t0 = time.perf_counter()
#     delta = dp_fixed(delta, z) if dp_mode == "fixed" else dp_adaptive(delta, history, z)
#     if DEV.type == "cuda":
#         torch.cuda.synchronize()
#     timer.append(time.perf_counter() - t0)
#     return delta


# def evaluate(model, loader):
#     model.eval(); c = t = 0
#     with torch.no_grad():
#         for xb, yb in loader:
#             xb, yb = xb.to(DEV), yb.to(DEV)
#             c += (model(xb).argmax(1) == yb).sum().item(); t += yb.size(0)
#     return c / t


# def run_fl(clients, test_loader, dp_mode, z, snapshot_rounds=()):
#     model = LeNet().to(DEV); model.apply(weights_init)
#     history = [[] for _ in model.parameters()]
#     timer, acc_curve, snaps = [], [], {}

#     for r in range(1, ROUNDS + 1):
#         deltas = [local_train(model, c, dp_mode, z, history, timer) for c in clients]
#         with torch.no_grad():
#             for i, p in enumerate(model.parameters()):
#                 p += torch.stack([d[i] for d in deltas]).mean(0)
#         acc = evaluate(model, test_loader); acc_curve.append(acc)
#         if r in snapshot_rounds:
#             snaps[r] = copy.deepcopy(model)
#         print(f"  [{dp_mode} z={z}] round {r:2d}  acc {acc:.4f}", flush=True)
#     return model, acc_curve, snaps, timer


# # =============================================================================
# # ATTACK
# # =============================================================================
# def dlg_attack(model, target, shape, iters=ATTACK_ITERS, label=""):
#     torch.manual_seed(SEED)
#     dx = torch.randn(shape, device=DEV).requires_grad_(True)
#     dy = torch.randn((1, N_CLASS), device=DEV).requires_grad_(True)
#     opt = torch.optim.LBFGS([dx, dy], lr=1.0)
#     target = [t.detach() for t in target]

#     for it in range(iters):
#         def closure():
#             opt.zero_grad()
#             loss = torch.mean(torch.sum(
#                 -torch.softmax(dy, -1) * torch.log_softmax(model(dx), -1), 1))
#             dg = torch.autograd.grad(loss, model.parameters(), create_graph=True)
#             diff = sum(((a - b) ** 2).sum() for a, b in zip(dg, target))
#             diff.backward(); return diff
#         try:
#             opt.step(closure)
#         except Exception as e:
#             print(f"    [{label}] stopped at {it}: {e}"); break
#         if it % 100 == 0:
#             print(f"    [{label}] iter {it}", flush=True)
#     return dx.detach()


# def victim_gradient(model, x, y):
#     loss = nn.CrossEntropyLoss()(model(x), y)
#     return [g.detach().clone() for g in torch.autograd.grad(loss, model.parameters())]


# def to_img(t):
#     a = t[0].detach().cpu().numpy().transpose(1, 2, 0)
#     return np.clip(a[:, :, 0] if a.shape[2] == 1 else a, 0, 1)


# def score(orig, rec):
#     o, r = to_img(orig), to_img(rec)
#     mse = float(np.mean((o - r) ** 2))
#     s = float(ssim_fn(o, r, channel_axis=-1 if o.ndim == 3 else None, data_range=1.0))
#     return mse, s


# # =============================================================================
# # MAIN
# # =============================================================================
# def main():
#     clients, test_loader, train_full = load_data()
#     vx = train_full[VICTIM_IDX][0].unsqueeze(0).to(DEV)
#     vy = torch.tensor([train_full[VICTIM_IDX][1]], device=DEV)

#     # ---- diagnostic: will these z values actually produce a gradient? ----
#     probe = LeNet().to(DEV)
#     d = sum(p.numel() for p in probe.parameters())
#     print(f"\n--- diagnostic ---")
#     print(f"model parameters d = {d},  sqrt(d) = {d**0.5:.1f}")
#     print("noise/signal ratio ~ z * sqrt(d) / K   (K = n_clients averaging)")
#     for z in Z_LIST:
#         print(f"  z={z:<8} ratio ~ {z * d**0.5 / N_CLIENTS:7.3f}")
#     print("  want roughly 0.5 -> 10 across your z values\n")

#     print("=== Arm A: FedAvg (no DP) ===")
#     _, accA, snapsA, _ = run_fl(clients, test_loader, "none", None,
#                                 snapshot_rounds=(1, ROUNDS))

#     print(f"\n=== Arm B: fixed clipping, z={Z_UTILITY} ===")
#     _, accB, _, timeB = run_fl(clients, test_loader, "fixed", Z_UTILITY)

#     print(f"\n=== Arm C: adaptive clipping, z={Z_UTILITY} ===")
#     _, accC, _, timeC = run_fl(clients, test_loader, "adaptive", Z_UTILITY)

#     # ---------------- EXP 1 : attack at round 10 ----------------
#     print("\n=== EXP 1: attack at round 10 ===")
#     m10 = snapsA[ROUNDS]
#     g = victim_gradient(m10, vx, vy)
#     gnorm = torch.sqrt(sum((t**2).sum() for t in g)).item()
#     print(f"  victim gradient L2 norm = {gnorm:.4f}  (clip C = {FIXED_CLIP_C})")

#     hist = [[t.norm().item()] * 3 for t in g]
#     runs = [("No DP", g)]
#     runs += [(f"Fixed z={z}", dp_fixed(g, z)) for z in Z_LIST]
#     runs += [(f"Adapt z={z}", dp_adaptive(g, [h[:] for h in hist], z)) for z in Z_LIST]

#     results, images = [], []
#     for nm, gg in runs:
#         print(f"  -> {nm}")
#         rec = dlg_attack(m10, gg, vx.shape, label=nm)
#         mse, s = score(vx, rec)
#         results.append((nm, mse, s)); images.append((nm, rec, s))
#         print(f"     MSE {mse:.4f}  SSIM {s:.4f}")

#     # ---------------- EXP 2 : round 1 vs round 10 ----------------
#     print("\n=== EXP 2: training stage ===")
#     m1 = snapsA[1]
#     rec1 = dlg_attack(m1, victim_gradient(m1, vx, vy), vx.shape, label="r1")
#     mse1, s1 = score(vx, rec1)
#     print(f"  round 1 : MSE {mse1:.4f}  SSIM {s1:.4f}")
#     print(f"  round 10: MSE {results[0][1]:.4f}  SSIM {results[0][2]:.4f}")

#     # ---------------- EXP 3 : accumulated update ----------------
#     print("\n=== EXP 3: accumulated multi-epoch update (no DP) ===")
#     tmp = copy.deepcopy(m10).to(DEV)
#     opt = torch.optim.SGD(tmp.parameters(), lr=LR)
#     crit = nn.CrossEntropyLoss()
#     loader = DataLoader(clients[0], batch_size=BATCH_SIZE, shuffle=True)
#     start = [p.detach().clone() for p in m10.parameters()]
#     steps = 0
#     for _ in range(LOCAL_EPOCHS):
#         for xb, yb in loader:
#             xb, yb = xb.to(DEV), yb.to(DEV)
#             opt.zero_grad(); crit(tmp(xb), yb).backward(); opt.step(); steps += 1
#     delta = [p.detach() - s for p, s in zip(tmp.parameters(), start)]
#     g_acc = [-dd / (LR * steps) for dd in delta]
#     rec_a = dlg_attack(m10, g_acc, vx.shape, label="accum")
#     mse_a, s_a = score(vx, rec_a)
#     print(f"  accumulated ({steps} steps): MSE {mse_a:.4f}  SSIM {s_a:.4f}")

#     # ---------------- OUTPUT ----------------
#     L = ["=" * 60,
#          f"{DATASET}  clients={N_CLIENTS} rounds={ROUNDS} "
#          f"local_epochs={LOCAL_EPOCHS} batch={BATCH_SIZE}",
#          f"parameterised by noise multiplier z (NOT epsilon)",
#          "=" * 60,
#          "\nEXP 1 -- reconstruction quality (round 10)",
#          f"{'Setting':<20}{'MSE':>12}{'SSIM':>12}", "-" * 44]
#     L += [f"{n:<20}{m:>12.4f}{s:>12.4f}" for n, m, s in results]
#     L += ["\nEXP 2 -- training stage",
#           f"{'Round 1':<20}{mse1:>12.4f}{s1:>12.4f}",
#           f"{'Round 10':<20}{results[0][1]:>12.4f}{results[0][2]:>12.4f}",
#           "\nEXP 3 -- accumulated update (negative result)",
#           f"{'Accumulated':<20}{mse_a:>12.4f}{s_a:>12.4f}",
#           "\nEXP 4 -- final test accuracy",
#           f"{'No DP':<20}{accA[-1]:>12.4f}",
#           f"{'Fixed z=' + str(Z_UTILITY):<20}{accB[-1]:>12.4f}",
#           f"{'Adapt z=' + str(Z_UTILITY):<20}{accC[-1]:>12.4f}",
#           "\nEXP 5 -- DP overhead per call",
#           f"{'Fixed':<20}{np.mean(timeB)*1e6:>10.1f} us",
#           f"{'Adaptive':<20}{np.mean(timeC)*1e6:>10.1f} us", "=" * 60]
#     out = "\n".join(L); print("\n" + out)
#     open("results_table.txt", "w").write(out)

#     n = len(images) + 1
#     fig, ax = plt.subplots(1, n, figsize=(2.2 * n, 3.0))
#     cm = None if IN_CH == 3 else "gray"
#     ax[0].imshow(to_img(vx), cmap=cm); ax[0].set_title("Original", fontsize=9); ax[0].axis("off")
#     for i, (nm, r, s) in enumerate(images):
#         ax[i+1].imshow(to_img(r), cmap=cm)
#         ax[i+1].set_title(f"{nm}\nSSIM {s:.3f}", fontsize=8); ax[i+1].axis("off")
#     plt.tight_layout(); plt.savefig("fig_reconstructions.png", dpi=150, bbox_inches="tight")
#     plt.show()

#     plt.figure(figsize=(6, 4)); rr = range(1, ROUNDS + 1)
#     plt.plot(rr, accA, "o-", label="No DP")
#     plt.plot(rr, accB, "s-", label=f"Fixed z={Z_UTILITY}")
#     plt.plot(rr, accC, "^-", label=f"Adaptive z={Z_UTILITY}")
#     plt.xlabel("Round"); plt.ylabel("Test accuracy"); plt.legend(); plt.grid(alpha=0.3)
#     plt.title(f"Utility cost of DP ({DATASET})")
#     plt.tight_layout(); plt.savefig("fig_accuracy.png", dpi=150); plt.show()
#     print("\nSaved figures and results_table.txt")


# if __name__ == "__main__":
#     main()

In [8]:
# """
# =============================================================================
# FL + DP vs Gradient Inversion  --  v4
# =============================================================================

# Changes from v3:
#   1. SiLU activation everywhere. Smooth (so DLG's second derivative exists)
#      but trains far better than sigmoid. ONE model for both utility and
#      attack -- accuracy and SSIM must come from the same system or the
#      privacy-utility comparison is meaningless.
#   2. CIFAR normalisation + augmentation on training data.
#      Victim image uses DETERMINISTIC preprocessing (no crop/flip) -- otherwise
#      you attack a different image than the one you display.
#   3. Images are unnormalised before MSE/SSIM/display.
#   4. SGD with momentum. NOT Adam: Adam's per-coordinate moment rescaling
#      breaks the delta->gradient relationship, which would invalidate EXP 3.
#   5. Batch 128 (matches Cui & Wu).
#   6. SMOKE_TEST flag -- run 2 rounds first to verify accuracy climbs.

# Epsilon note: at this scale (aggregate-level clipping, no per-example
# clipping, no subsampling amplification) the transition sits at larger eps
# than the conventional [1,10] range. The diagnostic prints where your
# EPS_LIST lands. See limitations discussion.
# =============================================================================
# """

# import time, copy
# import numpy as np
# import torch
# import torch.nn as nn
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt
# from torchvision import datasets, transforms
# from torch.utils.data import DataLoader, Subset
# from skimage.metrics import structural_similarity as ssim_fn

# # =============================================================================
# # CONFIG
# # =============================================================================
# SMOKE_TEST    = False         # True -> 2 rounds, arm A only, one attack
# DATASET       = "CIFAR10"
# N_CLIENTS     = 10
# ROUNDS        = 30
# LOCAL_EPOCHS  = 2
# BATCH_SIZE    = 128
# LR            = 0.05
# MOMENTUM      = 0.9
# SAMPLES_PER_CLIENT = 2000
# FIXED_CLIP_C  = 1.0
# ADAPTIVE_ETA  = 0.5
# DELTA         = 1e-5

# EPS_LIST      = [200, 60, 20]   # CHECK DIAGNOSTIC BEFORE COMMITTING
# EPS_UTILITY   = 60

# ATTACK_ITERS  = 200
# VICTIM_IDX    = 25
# SEED          = 0

# if SMOKE_TEST:
#     ROUNDS = 2

# torch.manual_seed(SEED); np.random.seed(SEED)
# DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"device: {DEV}")

# IN_CH, IMG_SIZE, N_CLASS = 3, 32, 10
# MEAN = (0.4914, 0.4822, 0.4465)
# STD  = (0.2470, 0.2435, 0.2616)
# _MEAN_T = torch.tensor(MEAN).view(1, 3, 1, 1)
# _STD_T  = torch.tensor(STD).view(1, 3, 1, 1)


# def unnormalise(t):
#     """Undo normalisation before display / SSIM / MSE."""
#     m, s = _MEAN_T.to(t.device), _STD_T.to(t.device)
#     return (t * s + m).clamp(0, 1)


# # =============================================================================
# # MODEL -- SiLU: smooth enough for DLG, trains far better than sigmoid
# # =============================================================================
# class CNN(nn.Module):
#     def __init__(self):
#         super().__init__()
#         act = nn.SiLU
#         self.body = nn.Sequential(
#             nn.Conv2d(IN_CH, 32, 5, padding=2, stride=2), act(),
#             nn.Conv2d(32, 64, 5, padding=2, stride=2), act(),
#             nn.Conv2d(64, 64, 5, padding=2, stride=1), act(),
#         )
#         self.fc = nn.Linear(64 * 8 * 8, N_CLASS)

#     def forward(self, x):
#         out = self.body(x)
#         return self.fc(out.view(out.size(0), -1))


# def weights_init(m):
#     if isinstance(m, (nn.Conv2d, nn.Linear)):
#         nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
#         if m.bias is not None:
#             nn.init.zeros_(m.bias)


# # =============================================================================
# # DATA -- augmented for training, deterministic for the victim
# # =============================================================================
# def load_data():
#     train_tf = transforms.Compose([
#         transforms.RandomCrop(32, padding=4),
#         transforms.RandomHorizontalFlip(),
#         transforms.ToTensor(),
#         transforms.Normalize(MEAN, STD),
#     ])
#     eval_tf = transforms.Compose([
#         transforms.ToTensor(),
#         transforms.Normalize(MEAN, STD),
#     ])

#     train = datasets.CIFAR10("./data", train=True,  download=True, transform=train_tf)
#     victim_src = datasets.CIFAR10("./data", train=True, download=True, transform=eval_tf)
#     test  = datasets.CIFAR10("./data", train=False, download=True, transform=eval_tf)

#     idx = np.random.permutation(len(train))
#     clients = [Subset(train, idx[c*SAMPLES_PER_CLIENT:(c+1)*SAMPLES_PER_CLIENT].tolist())
#                for c in range(N_CLIENTS)]
#     test_loader = DataLoader(Subset(test, list(range(2000))), batch_size=256)
#     return clients, test_loader, victim_src


# # =============================================================================
# # DP MECHANISMS
# # =============================================================================
# def gaussian_sigma(sensitivity, eps, delta=DELTA):
#     return sensitivity * np.sqrt(2.0 * np.log(1.25 / delta)) / eps


# def dp_fixed(tensors, eps, C=FIXED_CLIP_C):
#     """Arm B -- LDP-Fed style: one global clipping threshold + Gaussian."""
#     total = torch.sqrt(sum((t ** 2).sum() for t in tensors))
#     scale = min(1.0, C / (total.item() + 1e-12))
#     sigma = gaussian_sigma(C, eps)
#     return [t * scale + torch.randn_like(t) * sigma for t in tensors]


# def dp_adaptive(tensors, history, eps, eta=ADAPTIVE_ETA):
#     """
#     Arm C -- ALDP-FL inspired:
#       per-layer threshold = mean of last 3 L2 norms (sliding window k=3)
#       Gaussian noise truncated to [-b, b]
#     NOTE: the threshold is computed from UNPRIVATISED gradient norms, so this
#     arm is ALDP-FL-inspired rather than formally (eps,delta)-DP as written.
#     """
#     b = (np.exp(min(eps, 20)) - eta) / (np.exp(min(eps, 20)) + eta)
#     out = []
#     for i, t in enumerate(tensors):
#         n = t.norm().item()
#         history[i].append(n)
#         history[i][:] = history[i][-3:]
#         C_layer = float(np.mean(history[i])) + 1e-12
#         scale = min(1.0, C_layer / (n + 1e-12))
#         noise = torch.clamp(torch.randn_like(t) * gaussian_sigma(C_layer, eps), -b, b)
#         out.append(t * scale + noise)
#     return out


# # =============================================================================
# # TRAINING
# # =============================================================================
# def local_train(global_model, dataset, dp_mode, eps, history, timer):
#     model = copy.deepcopy(global_model).to(DEV)
#     opt = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM)
#     crit = nn.CrossEntropyLoss()
#     loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
#     start = [p.detach().clone() for p in global_model.parameters()]

#     model.train()
#     for _ in range(LOCAL_EPOCHS):
#         for xb, yb in loader:
#             xb, yb = xb.to(DEV), yb.to(DEV)
#             opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()

#     delta = [p.detach() - s for p, s in zip(model.parameters(), start)]
#     if dp_mode == "none":
#         return delta

#     if DEV.type == "cuda": torch.cuda.synchronize()
#     t0 = time.perf_counter()
#     delta = dp_fixed(delta, eps) if dp_mode == "fixed" else dp_adaptive(delta, history, eps)
#     if DEV.type == "cuda": torch.cuda.synchronize()
#     timer.append(time.perf_counter() - t0)
#     return delta


# def evaluate(model, loader):
#     model.eval(); c = t = 0
#     with torch.no_grad():
#         for xb, yb in loader:
#             xb, yb = xb.to(DEV), yb.to(DEV)
#             c += (model(xb).argmax(1) == yb).sum().item(); t += yb.size(0)
#     return c / t


# def run_fl(clients, test_loader, dp_mode, eps, init_state, snapshot_rounds=()):
#     """All arms start from IDENTICAL weights -- required for a fair comparison."""
#     model = CNN().to(DEV)
#     model.load_state_dict(copy.deepcopy(init_state))
#     history = [[] for _ in model.parameters()]
#     timer, acc_curve, snaps = [], [], {}

#     for r in range(1, ROUNDS + 1):
#         deltas = [local_train(model, c, dp_mode, eps, history, timer) for c in clients]
#         with torch.no_grad():
#             for i, p in enumerate(model.parameters()):
#                 p += torch.stack([d[i] for d in deltas]).mean(0)
#         acc = evaluate(model, test_loader); acc_curve.append(acc)
#         if r in snapshot_rounds:
#             snaps[r] = copy.deepcopy(model)
#         if r % 5 == 0 or r <= 2:
#             print(f"  [{dp_mode} eps={eps}] round {r:2d}  acc {acc:.4f}", flush=True)
#     return model, acc_curve, snaps, timer


# # =============================================================================
# # ATTACK
# # =============================================================================
# def dlg_attack(model, target, shape, iters=ATTACK_ITERS, label=""):
#     torch.manual_seed(SEED)
#     dx = torch.randn(shape, device=DEV).requires_grad_(True)
#     dy = torch.randn((1, N_CLASS), device=DEV).requires_grad_(True)
#     opt = torch.optim.LBFGS([dx, dy], lr=1.0)
#     target = [t.detach() for t in target]

#     for it in range(iters):
#         def closure():
#             opt.zero_grad()
#             loss = torch.mean(torch.sum(
#                 -torch.softmax(dy, -1) * torch.log_softmax(model(dx), -1), 1))
#             dg = torch.autograd.grad(loss, model.parameters(), create_graph=True)
#             diff = sum(((a - b) ** 2).sum() for a, b in zip(dg, target))
#             diff.backward(); return diff
#         try:
#             opt.step(closure)
#         except Exception as e:
#             print(f"    [{label}] stopped at {it}: {e}"); break
#         if it % 100 == 0:
#             print(f"    [{label}] iter {it}", flush=True)
#     return dx.detach()


# def victim_gradient(model, x, y):
#     loss = nn.CrossEntropyLoss()(model(x), y)
#     return [g.detach().clone() for g in torch.autograd.grad(loss, model.parameters())]


# def to_img(t):
#     a = unnormalise(t)[0].detach().cpu().numpy().transpose(1, 2, 0)
#     return np.clip(a, 0, 1)


# def score(orig, rec):
#     o, r = to_img(orig), to_img(rec)
#     mse = float(np.mean((o - r) ** 2))
#     s = float(ssim_fn(o, r, channel_axis=-1, data_range=1.0))
#     return mse, s


# # =============================================================================
# # MAIN
# # =============================================================================
# def main():
#     clients, test_loader, victim_src = load_data()
#     vx = victim_src[VICTIM_IDX][0].unsqueeze(0).to(DEV)      # deterministic
#     vy = torch.tensor([victim_src[VICTIM_IDX][1]], device=DEV)

#     torch.manual_seed(SEED)
#     base = CNN().to(DEV); base.apply(weights_init)
#     init_state = copy.deepcopy(base.state_dict())            # shared init

#     d = sum(p.numel() for p in base.parameters())
#     print("\n" + "-" * 58)
#     print(f"DIAGNOSTIC | d={d}  sqrt(d)={d**0.5:.1f}  K={N_CLIENTS}")
#     print("noise/signal ~ 4.85*sqrt(d)/(eps*K)")
#     print("-" * 58)
#     for e in EPS_LIST:
#         ratio = 4.85 * (d ** 0.5) / (e * N_CLIENTS)
#         v = "too weak" if ratio < 0.3 else "good" if ratio < 8 else "saturated"
#         print(f"  eps={e:<6} sigma={gaussian_sigma(FIXED_CLIP_C,e):8.4f}"
#               f"  ratio={ratio:7.2f}  {v}")
#     print("-" * 58 + "\nwant ratios spanning ~0.3 to ~8\n")

#     # ---------------- SMOKE TEST ----------------
#     if SMOKE_TEST:
#         print("=== SMOKE TEST: 2 rounds, no DP ===")
#         _, acc, snaps, _ = run_fl(clients, test_loader, "none", None,
#                                   init_state, snapshot_rounds=(2,))
#         print(f"\naccuracy: {acc}")
#         print("-> should be clearly above 0.10 and increasing\n")
#         m = snaps[2]
#         rec = dlg_attack(m, victim_gradient(m, vx, vy), vx.shape, label="smoke")
#         mse, s = score(vx, rec)
#         print(f"reconstruction: MSE {mse:.4f}  SSIM {s:.4f}")
#         print("-> SSIM should be > 0.7. if not, try SiLU->Sigmoid or more iters")
#         fig, ax = plt.subplots(1, 2, figsize=(5, 3))
#         ax[0].imshow(to_img(vx)); ax[0].set_title("Original"); ax[0].axis("off")
#         ax[1].imshow(to_img(rec)); ax[1].set_title(f"SSIM {s:.3f}"); ax[1].axis("off")
#         plt.tight_layout(); plt.savefig("smoke_test.png", dpi=150); plt.show()
#         return

#     # ---------------- TRAINING ----------------
#     print("=== Arm A: FedAvg (no DP) ===")
#     _, accA, snapsA, _ = run_fl(clients, test_loader, "none", None,
#                                 init_state, snapshot_rounds=(1, ROUNDS))

#     print(f"\n=== Arm B: fixed clipping, eps={EPS_UTILITY} ===")
#     _, accB, _, timeB = run_fl(clients, test_loader, "fixed", EPS_UTILITY, init_state)

#     print(f"\n=== Arm C: adaptive clipping, eps={EPS_UTILITY} ===")
#     _, accC, _, timeC = run_fl(clients, test_loader, "adaptive", EPS_UTILITY, init_state)

#     # ---------------- EXP 1 ----------------
#     print(f"\n=== EXP 1: attack at round {ROUNDS} ===")
#     mF = snapsA[ROUNDS]
#     g = victim_gradient(mF, vx, vy)
#     print(f"  gradient L2 norm = {torch.sqrt(sum((t**2).sum() for t in g)).item():.4f}"
#           f"   (clip C = {FIXED_CLIP_C})")

#     hist = [[t.norm().item()] * 3 for t in g]
#     runs = [("No DP", g)]
#     runs += [(f"Fixed eps={e}", dp_fixed(g, e)) for e in EPS_LIST]
#     runs += [(f"Adapt eps={e}", dp_adaptive(g, [h[:] for h in hist], e)) for e in EPS_LIST]

#     results, images = [], []
#     for nm, gg in runs:
#         print(f"  -> {nm}")
#         rec = dlg_attack(mF, gg, vx.shape, label=nm)
#         mse, s = score(vx, rec)
#         results.append((nm, mse, s)); images.append((nm, rec, s))
#         print(f"     MSE {mse:.4f}  SSIM {s:.4f}")

#     # ---------------- EXP 2 ----------------
#     print("\n=== EXP 2: training stage ===")
#     m1 = snapsA[1]
#     rec1 = dlg_attack(m1, victim_gradient(m1, vx, vy), vx.shape, label="r1")
#     mse1, s1 = score(vx, rec1)
#     print(f"  round  1: MSE {mse1:.4f}  SSIM {s1:.4f}")
#     print(f"  round {ROUNDS}: MSE {results[0][1]:.4f}  SSIM {results[0][2]:.4f}")

#     # ---------------- EXP 3 ----------------
#     print("\n=== EXP 3: accumulated multi-epoch update (no DP) ===")
#     tmp = copy.deepcopy(mF).to(DEV)
#     opt = torch.optim.SGD(tmp.parameters(), lr=LR, momentum=MOMENTUM)
#     crit = nn.CrossEntropyLoss()
#     loader = DataLoader(clients[0], batch_size=BATCH_SIZE, shuffle=True)
#     start = [p.detach().clone() for p in mF.parameters()]
#     steps = 0
#     for _ in range(LOCAL_EPOCHS):
#         for xb, yb in loader:
#             xb, yb = xb.to(DEV), yb.to(DEV)
#             opt.zero_grad(); crit(tmp(xb), yb).backward(); opt.step(); steps += 1
#     delta = [p.detach() - s for p, s in zip(tmp.parameters(), start)]
#     g_acc = [-dd / (LR * steps) for dd in delta]
#     rec_a = dlg_attack(mF, g_acc, vx.shape, label="accum")
#     mse_a, s_a = score(vx, rec_a)
#     print(f"  accumulated ({steps} steps): MSE {mse_a:.4f}  SSIM {s_a:.4f}")

#     # ---------------- OUTPUT ----------------
#     L = ["=" * 60,
#          f"CIFAR10  clients={N_CLIENTS}  rounds={ROUNDS}  "
#          f"local_epochs={LOCAL_EPOCHS}  batch={BATCH_SIZE}",
#          f"SiLU CNN, SGD(lr={LR}, mom={MOMENTUM}), normalised + augmented",
#          f"Gaussian mechanism, delta={DELTA}, clip C={FIXED_CLIP_C}",
#          "=" * 60,
#          f"\nEXP 1 -- reconstruction quality (round {ROUNDS})",
#          f"{'Setting':<20}{'MSE':>12}{'SSIM':>12}", "-" * 44]
#     L += [f"{n:<20}{m:>12.4f}{s:>12.4f}" for n, m, s in results]
#     L += ["\nEXP 2 -- training stage",
#           f"{'Round 1':<20}{mse1:>12.4f}{s1:>12.4f}",
#           f"{'Round ' + str(ROUNDS):<20}{results[0][1]:>12.4f}{results[0][2]:>12.4f}",
#           "\nEXP 3 -- accumulated update (negative result)",
#           f"{'Accumulated':<20}{mse_a:>12.4f}{s_a:>12.4f}",
#           "\nEXP 4 -- final test accuracy",
#           f"{'No DP':<20}{accA[-1]:>12.4f}",
#           f"{'Fixed eps=' + str(EPS_UTILITY):<20}{accB[-1]:>12.4f}",
#           f"{'Adapt eps=' + str(EPS_UTILITY):<20}{accC[-1]:>12.4f}",
#           "\nEXP 5 -- DP overhead per call",
#           f"{'Fixed':<20}{np.mean(timeB)*1e6:>10.1f} us",
#           f"{'Adaptive':<20}{np.mean(timeC)*1e6:>10.1f} us", "=" * 60]
#     out = "\n".join(L); print("\n" + out)
#     open("results_table.txt", "w").write(out)

#     n = len(images) + 1
#     fig, ax = plt.subplots(1, n, figsize=(2.2 * n, 3.0))
#     ax[0].imshow(to_img(vx)); ax[0].set_title("Original", fontsize=9); ax[0].axis("off")
#     for i, (nm, r, s) in enumerate(images):
#         ax[i+1].imshow(to_img(r))
#         ax[i+1].set_title(f"{nm}\nSSIM {s:.3f}", fontsize=8); ax[i+1].axis("off")
#     plt.tight_layout(); plt.savefig("fig_reconstructions.png", dpi=150, bbox_inches="tight")
#     plt.show()

#     plt.figure(figsize=(6, 4)); rr = range(1, ROUNDS + 1)
#     plt.plot(rr, accA, "-", label="No DP")
#     plt.plot(rr, accB, "-", label=f"Fixed eps={EPS_UTILITY}")
#     plt.plot(rr, accC, "-", label=f"Adaptive eps={EPS_UTILITY}")
#     plt.xlabel("Communication round"); plt.ylabel("Test accuracy")
#     plt.legend(); plt.grid(alpha=0.3)
#     plt.title(f"Utility cost of DP (CIFAR-10, {N_CLIENTS} clients)")
#     plt.tight_layout(); plt.savefig("fig_accuracy.png", dpi=150); plt.show()
#     print("\nSaved figures and results_table.txt")


# if __name__ == "__main__":
#     main()

In [9]:
"""
=============================================================================
FL + DP vs Gradient Inversion  --  v5
=============================================================================

FIXES FROM v4 (in order of importance):

 1. ATTACK CHECKPOINT. v4 attacked the round-30 model and the no-DP attack
    FAILED (SSIM -0.004). Plain DLG cannot invert a large, well-converged
    network -- that is exactly what Geiping et al. (2020) had to solve with
    cosine loss + TV priors. v4's own EXP 2 showed round 1 gives SSIM 1.000.
    We now attack an EARLY checkpoint and report the late one as a finding.

 2. PER-CLIENT HISTORIES. v4 shared one sliding window across all 10 clients,
    so "the last 3 norms" were from 3 DIFFERENT clients. Now each client
    keeps its own history, which is what client-local adaptation means.

 3. ADAPTIVE CLIPPING NOW ACTUALLY CLIPS. v4 seeded the window with the
    CURRENT gradient norm, giving C_layer = ||g_layer|| and scale = 1.0, so
    no clipping ever occurred and sigma grew without bound (accuracy 0.098 =
    random). The window is now bootstrapped from a fixed value and only ever
    contains PAST norms.

 4. DIAGNOSTIC SCALING. Averaging K independent noise draws reduces the
    standard deviation by sqrt(K), not K. v4 understated aggregate noise by
    ~3.2x, which is why eps=60 looked "good" and then trained at 36%.

 5. EXP 3 USES PLAIN SGD. With momentum, -delta/(lr*S) recovers the average
    VELOCITY, not the average gradient. EXP 3 now runs its own momentum-free
    local training so the inversion is mathematically what we claim.

 6. HONEST LABELLING. sigma = C*sqrt(2 ln(1.25/delta))/eps is a PER-RELEASE
    Gaussian calibration. No composition accounting is performed across
    rounds, and sensitivity under update-replacement may be 2C rather than C.
    Output labels these as per-release noise settings, not trained-model eps.

 7. Total round wall-clock time is measured, not only the DP call.
=============================================================================
"""

import time, copy
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from skimage.metrics import structural_similarity as ssim_fn

# =============================================================================
# CONFIG
# =============================================================================
SMOKE_TEST    = False
N_CLIENTS     = 10
ROUNDS        = 20
LOCAL_EPOCHS  = 2
BATCH_SIZE    = 128
LR            = 0.05
MOMENTUM      = 0.9
SAMPLES_PER_CLIENT = 2000

FIXED_CLIP_C  = 1.0
BOOTSTRAP_C   = 0.5           # seeds the adaptive window before any history
NOISE_BOUND   = 1.0
DELTA         = 1e-5

# per-release noise settings (NOT accounted total epsilon -- see header)
EPS_LIST      = [600, 200, 60]
EPS_UTILITY   = [600, 200]    # utility curves are run for each of these

ATTACK_ROUND  = 1             # <-- THE KEY FIX. see header item 1.
LATE_ROUND    = ROUNDS
ATTACK_ITERS  = 200
VICTIM_IDX    = 25
SEED          = 0

if SMOKE_TEST:
    ROUNDS = 2; LATE_ROUND = 2; EPS_UTILITY = [200]

torch.manual_seed(SEED); np.random.seed(SEED)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEV}")

N_CLASS = 10
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
_M = torch.tensor(MEAN).view(1, 3, 1, 1)
_S = torch.tensor(STD).view(1, 3, 1, 1)


def unnormalise(t):
    return (t * _S.to(t.device) + _M.to(t.device)).clamp(0, 1)


# =============================================================================
# MODEL
# =============================================================================
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        act = nn.SiLU
        self.body = nn.Sequential(
            nn.Conv2d(3, 32, 5, padding=2, stride=2), act(),
            nn.Conv2d(32, 64, 5, padding=2, stride=2), act(),
            nn.Conv2d(64, 64, 5, padding=2, stride=1), act(),
        )
        self.fc = nn.Linear(64 * 8 * 8, N_CLASS)

    def forward(self, x):
        o = self.body(x)
        return self.fc(o.view(o.size(0), -1))


def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        if m.bias is not None:
            nn.init.zeros_(m.bias)


# =============================================================================
# DATA
# =============================================================================
def load_data():
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    eval_tf = transforms.Compose([
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

    train = datasets.CIFAR10("./data", train=True, download=True, transform=train_tf)
    victim = datasets.CIFAR10("./data", train=True, download=True, transform=eval_tf)
    test = datasets.CIFAR10("./data", train=False, download=True, transform=eval_tf)

    idx = np.random.permutation(len(train))
    clients = [Subset(train, idx[c*SAMPLES_PER_CLIENT:(c+1)*SAMPLES_PER_CLIENT].tolist())
               for c in range(N_CLIENTS)]
    return clients, DataLoader(Subset(test, list(range(2000))), batch_size=256), victim


# =============================================================================
# DP MECHANISMS
# =============================================================================
def gaussian_sigma(sens, eps, delta=DELTA):
    """Per-release Gaussian calibration. No composition accounting."""
    return sens * np.sqrt(2.0 * np.log(1.25 / delta)) / eps


def dp_fixed(tensors, eps, C=FIXED_CLIP_C):
    """Arm B -- LDP-Fed style: one global threshold + plain Gaussian."""
    total = torch.sqrt(sum((t ** 2).sum() for t in tensors))
    scale = min(1.0, C / (total.item() + 1e-12))
    sigma = gaussian_sigma(C, eps)
    return [t * scale + torch.randn_like(t) * sigma for t in tensors]


def dp_adaptive(tensors, history, eps, b=NOISE_BOUND):
    """
    Arm C -- ALDP-FL inspired.
    The window holds only PAST norms: the threshold is computed BEFORE the
    current norm is appended. Otherwise C_layer == ||g_layer||, scale == 1,
    and no clipping happens (this was the v4 bug).

    NOTE: the threshold derives from unprivatised gradient norms, so this arm
    is ALDP-FL-INSPIRED, not formally (eps,delta)-DP as implemented.
    """
    out = []
    for i, t in enumerate(tensors):
        C_layer = float(np.mean(history[i])) if history[i] else BOOTSTRAP_C
        C_layer = max(C_layer, 1e-8)
        n = t.norm().item()
        scale = min(1.0, C_layer / (n + 1e-12))
        noise = torch.clamp(torch.randn_like(t) * gaussian_sigma(C_layer, eps), -b, b)
        out.append(t * scale + noise)
        history[i].append(n)                      # append AFTER using it
        history[i][:] = history[i][-3:]           # sliding window k = 3
    return out


# =============================================================================
# TRAINING
# =============================================================================
def local_train(global_model, dataset, dp_mode, eps, history, dp_timer):
    model = copy.deepcopy(global_model).to(DEV)
    opt = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM)
    crit = nn.CrossEntropyLoss()
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    start = [p.detach().clone() for p in global_model.parameters()]

    model.train()
    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb, yb = xb.to(DEV), yb.to(DEV)
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()

    delta = [p.detach() - s for p, s in zip(model.parameters(), start)]
    if dp_mode == "none":
        return delta

    if DEV.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    delta = dp_fixed(delta, eps) if dp_mode == "fixed" else dp_adaptive(delta, history, eps)
    if DEV.type == "cuda": torch.cuda.synchronize()
    dp_timer.append(time.perf_counter() - t0)
    return delta


def evaluate(model, loader):
    model.eval(); c = t = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEV), yb.to(DEV)
            c += (model(xb).argmax(1) == yb).sum().item(); t += yb.size(0)
    return c / t


def run_fl(clients, test_loader, dp_mode, eps, init_state, snapshot_rounds=()):
    model = CNN().to(DEV)
    model.load_state_dict(copy.deepcopy(init_state))
    # FIX 2: one history per client
    histories = [[[] for _ in model.parameters()] for _ in range(N_CLIENTS)]
    dp_timer, round_timer, acc_curve, snaps = [], [], [], {}

    for r in range(1, ROUNDS + 1):
        if DEV.type == "cuda": torch.cuda.synchronize()
        r0 = time.perf_counter()
        deltas = [local_train(model, clients[c], dp_mode, eps, histories[c], dp_timer)
                  for c in range(N_CLIENTS)]
        with torch.no_grad():
            for i, p in enumerate(model.parameters()):
                p += torch.stack([d[i] for d in deltas]).mean(0)
        if DEV.type == "cuda": torch.cuda.synchronize()
        round_timer.append(time.perf_counter() - r0)

        acc = evaluate(model, test_loader); acc_curve.append(acc)
        if r in snapshot_rounds:
            snaps[r] = copy.deepcopy(model)
        if r % 5 == 0 or r <= 2:
            print(f"  [{dp_mode} eps={eps}] round {r:2d}  acc {acc:.4f}", flush=True)
    return model, acc_curve, snaps, dp_timer, round_timer


# =============================================================================
# ATTACK  (DLG -- dummy label is optimised, not analytically inferred)
# =============================================================================
def dlg_attack(model, target, shape, iters=ATTACK_ITERS, label=""):
    torch.manual_seed(SEED)
    dx = torch.randn(shape, device=DEV).requires_grad_(True)
    dy = torch.randn((1, N_CLASS), device=DEV).requires_grad_(True)
    opt = torch.optim.LBFGS([dx, dy], lr=1.0)
    target = [t.detach() for t in target]

    for it in range(iters):
        def closure():
            opt.zero_grad()
            loss = torch.mean(torch.sum(
                -torch.softmax(dy, -1) * torch.log_softmax(model(dx), -1), 1))
            dg = torch.autograd.grad(loss, model.parameters(), create_graph=True)
            diff = sum(((a - b) ** 2).sum() for a, b in zip(dg, target))
            diff.backward(); return diff
        try:
            opt.step(closure)
        except Exception as e:
            print(f"    [{label}] stopped at {it}: {e}"); break
        if it % 100 == 0:
            print(f"    [{label}] iter {it}", flush=True)
    return dx.detach()


def victim_gradient(model, x, y):
    loss = nn.CrossEntropyLoss()(model(x), y)
    return [g.detach().clone() for g in torch.autograd.grad(loss, model.parameters())]


def to_img(t):
    return np.clip(unnormalise(t)[0].detach().cpu().numpy().transpose(1, 2, 0), 0, 1)


def score(o_t, r_t):
    o, r = to_img(o_t), to_img(r_t)
    return (float(np.mean((o - r) ** 2)),
            float(ssim_fn(o, r, channel_axis=-1, data_range=1.0)))


# =============================================================================
# MAIN
# =============================================================================
def main():
    clients, test_loader, victim_src = load_data()
    vx = victim_src[VICTIM_IDX][0].unsqueeze(0).to(DEV)
    vy = torch.tensor([victim_src[VICTIM_IDX][1]], device=DEV)

    torch.manual_seed(SEED)
    base = CNN().to(DEV); base.apply(weights_init)
    init_state = copy.deepcopy(base.state_dict())

    # ---- diagnostic (FIX 4: sqrt(K), not K) ----
    d = sum(p.numel() for p in base.parameters())
    print("\n" + "-" * 60)
    print(f"DIAGNOSTIC | d={d}  sqrt(d)={d**0.5:.1f}  K={N_CLIENTS}")
    print("noise/signal ~ 4.85*sqrt(d)/(eps*sqrt(K))")
    print("-" * 60)
    for e in sorted(set(EPS_LIST + list(EPS_UTILITY)), reverse=True):
        ratio = 4.85 * (d ** 0.5) / (e * (N_CLIENTS ** 0.5))
        v = "weak" if ratio < 0.3 else "good" if ratio < 8 else "SATURATED"
        print(f"  eps={e:<6} sigma={gaussian_sigma(FIXED_CLIP_C,e):8.4f}"
              f"  ratio={ratio:7.2f}  {v}")
    print("-" * 60 + "\nwant ratios spanning ~0.3 to ~8\n")

    # ---- smoke test ----
    if SMOKE_TEST:
        print("=== SMOKE TEST ===")
        _, acc, snaps, _, _ = run_fl(clients, test_loader, "none", None,
                                     init_state, snapshot_rounds=(1,))
        print(f"FedAvg acc: {acc}   (should rise above 0.10)")
        m = snaps[1]
        rec = dlg_attack(m, victim_gradient(m, vx, vy), vx.shape, label="smoke")
        mse, s = score(vx, rec)
        print(f"reconstruction @ round 1: MSE {mse:.4f}  SSIM {s:.4f}")
        print("-> SSIM must be > 0.7 or the whole demo fails")
        fig, ax = plt.subplots(1, 2, figsize=(5, 3))
        ax[0].imshow(to_img(vx)); ax[0].set_title("Original"); ax[0].axis("off")
        ax[1].imshow(to_img(rec)); ax[1].set_title(f"SSIM {s:.3f}"); ax[1].axis("off")
        plt.tight_layout(); plt.savefig("smoke_test.png", dpi=150); plt.show()
        return

    # ---- training ----
    print("=== Arm A: FedAvg (no DP) ===")
    _, accA, snapsA, _, rtA = run_fl(clients, test_loader, "none", None, init_state,
                                     snapshot_rounds=(ATTACK_ROUND, LATE_ROUND))

    util = {}
    for e in EPS_UTILITY:
        print(f"\n=== Fixed clipping, eps={e} ===")
        _, acc, _, dt, rt = run_fl(clients, test_loader, "fixed", e, init_state)
        util[("fixed", e)] = (acc, dt, rt)
        print(f"\n=== Adaptive clipping, eps={e} ===")
        _, acc, _, dt, rt = run_fl(clients, test_loader, "adaptive", e, init_state)
        util[("adaptive", e)] = (acc, dt, rt)

    # ---- EXP 1 : attack the EARLY checkpoint ----
    print(f"\n=== EXP 1: attack at round {ATTACK_ROUND} ===")
    mA = snapsA[ATTACK_ROUND]
    g = victim_gradient(mA, vx, vy)
    print(f"  gradient L2 norm = {torch.sqrt(sum((t**2).sum() for t in g)).item():.4f}")

    def fresh_hist():
        return [[BOOTSTRAP_C] for _ in g]

    runs = [("No DP", g)]
    runs += [(f"Fixed eps={e}", dp_fixed(g, e)) for e in EPS_LIST]
    runs += [(f"Adapt eps={e}", dp_adaptive(g, fresh_hist(), e)) for e in EPS_LIST]

    results, images = [], []
    for nm, gg in runs:
        print(f"  -> {nm}")
        rec = dlg_attack(mA, gg, vx.shape, label=nm)
        mse, s = score(vx, rec)
        results.append((nm, mse, s)); images.append((nm, rec, s))
        print(f"     MSE {mse:.4f}  SSIM {s:.4f}")

    # ---- EXP 2 : late checkpoint resists plain DLG ----
    print(f"\n=== EXP 2: same attack at round {LATE_ROUND} ===")
    mL = snapsA[LATE_ROUND]
    recL = dlg_attack(mL, victim_gradient(mL, vx, vy), vx.shape, label="late")
    mseL, sL = score(vx, recL)
    print(f"  round {LATE_ROUND}: MSE {mseL:.4f}  SSIM {sL:.4f}")

    # ---- EXP 3 : accumulated update, PLAIN SGD (FIX 5) ----
    print("\n=== EXP 3: accumulated multi-epoch update (plain SGD, no DP) ===")
    tmp = copy.deepcopy(mA).to(DEV)
    opt = torch.optim.SGD(tmp.parameters(), lr=LR)          # no momentum
    crit = nn.CrossEntropyLoss()
    loader = DataLoader(clients[0], batch_size=BATCH_SIZE, shuffle=True)
    start = [p.detach().clone() for p in mA.parameters()]
    steps = 0
    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb, yb = xb.to(DEV), yb.to(DEV)
            opt.zero_grad(); crit(tmp(xb), yb).backward(); opt.step(); steps += 1
    delta = [p.detach() - s for p, s in zip(tmp.parameters(), start)]
    g_acc = [-dd / (LR * steps) for dd in delta]
    rec_a = dlg_attack(mA, g_acc, vx.shape, label="accum")
    mse_a, s_a = score(vx, rec_a)
    print(f"  accumulated ({steps} steps): MSE {mse_a:.4f}  SSIM {s_a:.4f}")

    # ---- output ----
    L = ["=" * 64,
         f"CIFAR-10 | clients={N_CLIENTS} rounds={ROUNDS} "
         f"local_epochs={LOCAL_EPOCHS} batch={BATCH_SIZE}",
         f"SiLU CNN | SGD(lr={LR}, mom={MOMENTUM}) | normalised + augmented",
         "",
         "NOTE: eps values are PER-RELEASE Gaussian calibrations.",
         "No composition accounting across rounds is performed.",
         "Attack is DLG (dummy label optimised, not analytically inferred).",
         "Adaptive arm is ALDP-FL-INSPIRED: its threshold derives from",
         "unprivatised gradient norms and is not separately accounted.",
         "=" * 64,
         f"\nEXP 1 -- reconstruction at round {ATTACK_ROUND}",
         f"{'Setting':<20}{'MSE':>12}{'SSIM':>12}", "-" * 44]
    L += [f"{n:<20}{m:>12.4f}{s:>12.4f}" for n, m, s in results]
    L += [f"\nEXP 2 -- same attack at round {LATE_ROUND}",
          f"{'Round ' + str(ATTACK_ROUND):<20}{results[0][1]:>12.4f}{results[0][2]:>12.4f}",
          f"{'Round ' + str(LATE_ROUND):<20}{mseL:>12.4f}{sL:>12.4f}",
          "\nEXP 3 -- accumulated update, plain SGD",
          f"{'Accumulated':<20}{mse_a:>12.4f}{s_a:>12.4f}",
          "\nEXP 4 -- final test accuracy",
          f"{'No DP':<26}{accA[-1]:>10.4f}"]
    for k in sorted(util, key=lambda x: (-x[1], x[0])):
        L.append(f"{k[0] + ' eps=' + str(k[1]):<26}{util[k][0][-1]:>10.4f}")
    L += ["\nEXP 5 -- overhead", f"{'':<26}{'DP call':>12}{'full round':>14}",
          f"{'No DP':<26}{'-':>12}{np.mean(rtA):>13.3f}s"]
    for k in sorted(util, key=lambda x: (-x[1], x[0])):
        _, dt, rt = util[k]
        L.append(f"{k[0] + ' eps=' + str(k[1]):<26}"
                 f"{np.mean(dt)*1e6:>10.1f}us{np.mean(rt):>13.3f}s")
    L.append("=" * 64)
    out = "\n".join(L); print("\n" + out)
    open("results_table.txt", "w").write(out)

    # ---- figures ----
    n = len(images) + 1
    fig, ax = plt.subplots(1, n, figsize=(2.2 * n, 3.0))
    ax[0].imshow(to_img(vx)); ax[0].set_title("Original", fontsize=9); ax[0].axis("off")
    for i, (nm, r, s) in enumerate(images):
        ax[i+1].imshow(to_img(r))
        ax[i+1].set_title(f"{nm}\nSSIM {s:.3f}", fontsize=8); ax[i+1].axis("off")
    plt.tight_layout(); plt.savefig("fig_reconstructions.png", dpi=150, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(7, 4.5)); rr = range(1, ROUNDS + 1)
    plt.plot(rr, accA, "k-", lw=2, label="No DP")
    for k in sorted(util, key=lambda x: (-x[1], x[0])):
        plt.plot(rr, util[k][0], "-", label=f"{k[0]} eps={k[1]}")
    plt.xlabel("Communication round"); plt.ylabel("Test accuracy")
    plt.legend(fontsize=8); plt.grid(alpha=0.3)
    plt.title(f"Privacy-utility trade-off (CIFAR-10, {N_CLIENTS} clients)")
    plt.tight_layout(); plt.savefig("fig_accuracy.png", dpi=150); plt.show()
    print("\nSaved figures and results_table.txt")


if __name__ == "__main__":
    main()

device: cuda

------------------------------------------------------------
DIAGNOSTIC | d=197130  sqrt(d)=444.0  K=10
noise/signal ~ 4.85*sqrt(d)/(eps*sqrt(K))
------------------------------------------------------------
  eps=600    sigma=  0.0081  ratio=   1.13  good
  eps=200    sigma=  0.0242  ratio=   3.40  good
  eps=60     sigma=  0.0807  ratio=  11.35  SATURATED
------------------------------------------------------------
want ratios spanning ~0.3 to ~8

=== Arm A: FedAvg (no DP) ===
  [none eps=None] round  1  acc 0.4100
  [none eps=None] round  2  acc 0.4615
  [none eps=None] round  5  acc 0.5230
  [none eps=None] round 10  acc 0.5965
  [none eps=None] round 15  acc 0.6410
  [none eps=None] round 20  acc 0.6715

=== Fixed clipping, eps=600 ===
  [fixed eps=600] round  1  acc 0.2980
  [fixed eps=600] round  2  acc 0.3765
  [fixed eps=600] round  5  acc 0.4680
  [fixed eps=600] round 10  acc 0.5135
  [fixed eps=600] round 15  acc 0.5490
  [fixed eps=600] round 20  acc 0.5755

=